In [1]:
pip install dask[dataframe] --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.8 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2026.3.0
    Uninstalling dask-2026.3.0:
      Successfully uninstalled dask-2026.3.0


In [2]:
!pip install scikit-learn==1.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 53.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [3]:
!pip install xgboost --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 8.2 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.2.0
    Uninstalling xgboost-3.2.0:
      Successfully uninstalled xgboost-3.2.0


In [4]:
import pandas as pd
import numpy as np
import dask.dataframe as dask_data

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objs as go
from plotly.offline import init_notebook_mode, iplot, plot, download_plotlyjs

import sklearn
import matplotlib.dates as mdates
matplotlib.style.use('ggplot')

#A parse date variable to pass in the read_csv function later to take into account the date format
parse_date = lambda val : pd.datetime.strptime(val, '%y%m%d%H')

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# A sample dataset of 100k lines.
!ls '/content/drive/My Drive/train_data'

def parse_date(date):
    return pd.to_datetime(date, format='%y%m%d%H')

train_data = pd.read_csv('/content/drive/My Drive/train_data/filtered_train.csv', parse_dates = ['hour'], date_parser=parse_date, nrows = 100000)

train_data.info()

train_data_clicks = train_data[train_data['click']==1]

In [ ]:
train_data.describe()


In [ ]:
train_data.head()

# **Data EDA**

In [ ]:
# Click v/s No click distribution
train_data.groupby('click').size().plot(kind = 'bar')
rows = train_data.shape[0]
click_through_rate = train_data['click'].value_counts()/rows
click_through_rate

In [ ]:
# hour
train_data.hour.describe()

In [ ]:
train_data['hour'].describe()

In [ ]:
# banner position
"""Banner positions representing attractive and appealing designs that might highly
affect a user's behavior and in turn trigger their decision to click. Or not.
Hence making it an effective metric to predict clicks"""
train_data['banner_pos'].unique()

In [ ]:
# banner position & click relation
banner_temp =train_data[['banner_pos','click']].groupby(['banner_pos','click'])

banner_temp.size().unstack().plot(kind='bar',stacked=True, title='banner positions')

In [ ]:
train_data[['banner_pos','click']].groupby(['banner_pos']).count().sort_values('click',ascending=False)

In [ ]:
# click percent for each banner position
banner_df = pd.DataFrame()
banner_df['position'] = train_data_clicks[['banner_pos','click']].groupby(['banner_pos']).count().reset_index().sort_values('click',ascending=False)['banner_pos']
banner_df['pos_clicks'] = train_data_clicks[['banner_pos','click']].groupby(['banner_pos']).count().reset_index().sort_values('click',ascending=False)['click']
banner_df['total_impressions'] = train_data[['banner_pos','click']].groupby(['banner_pos']).count().reset_index().sort_values('click',ascending=False)['click']
banner_df['click_pct'] = 100*banner_df['pos_clicks']/banner_df['total_impressions']
banner_df

In [ ]:
banner_df.sort_values(ascending=False,by='click_pct')

In [ ]:
# Device type & click relation
device_temp = train_data[['device_type','click']].groupby(['device_type','click'])
device_temp.size().unstack().plot(kind='bar',stacked=True, title='device types')

In [ ]:
train_data[['device_type','click']].groupby(['device_type']).count().sort_values('click',ascending=False)

In [ ]:
train_data_clicks[['device_type','click']].groupby(['device_type','click']).size().unstack().plot(kind='bar',stacked=True, title='device types')

In [ ]:
# click percent for each device type
dev_type_df = train_data_clicks.groupby('device_type').agg({'click':'sum'}).reset_index()
dev_type_df_total_imp = train_data.groupby('device_type').agg({'click':'count'}).reset_index()
dev_type_df['total_impressions'] = dev_type_df_total_imp['click']
dev_type_df['success_pct'] = (dev_type_df['click']/dev_type_df['total_impressions'])*100
dev_type_df

In [ ]:
# App category and click relation
app_features = ['app_id', 'app_domain', 'app_category']
train_data.groupby('app_category').agg({'click':'sum'}).sort_values(by='click',ascending = False)

In [ ]:
# Clicks behavior across different app categories
train_app_category = train_data.groupby(['app_category', 'click']).size().unstack()
train_app_category.div(train_app_category.sum(axis=1), axis=0).plot(kind='bar', stacked=True, title="Intra-category CTR")

In [ ]:
# C1, C14-C21 features
features = ['C1', 'C14', 'C15', 'C16', 'C17', 'C18',
            'C20', 'C21']
train_data[features].astype('object').describe()

In [ ]:
# train_data.groupby(['C1', 'click']).size().unstack().plot(kind='bar', stacked=True, title='C1 histogram')
# train_data.groupby(['C15', 'click']).size().unstack().plot(kind='bar', stacked=True, title='C1 histogram')
# train_data.groupby(['C16', 'click']).size().unstack().plot(kind='bar', stacked=True, title='C1 histogram')
train_data.groupby(['C18', 'click']).size().unstack().plot(kind='bar', stacked=True, title='C1 histogram')

# **Data Pre-processing**

In [ ]:
# select features
model_features = ['banner_pos', 'site_category', 'device_conn_type', 'app_category','device_type', 'C14', 'C17']
model_target = 'click'
train_model = train_data[model_features+[model_target]].sample(frac=0.1)

In [ ]:
train_model.shape

In [ ]:
# categorical feature encoding
def one_hot_features(data_frame, feature_set):
    new_data_frame = pd.get_dummies(data_frame,
                                    columns = feature_set,
                                    sparse = True)

    return new_data_frame

In [ ]:
train_model = one_hot_features(train_model,
                                ['site_category',
                                 'app_category',
                                 'banner_pos'])
train_model.head()

In [ ]:
train_model.shape

In [ ]:
# Extracting all columns from the train model except the target mask column
model_features = np.array(train_model.columns[train_model.columns!=model_target].tolist())

In [ ]:
# train test split
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    train_model[model_features].values,
    train_model[model_target].values,
    test_size=0.3,
)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import f1_score

# F1 score used as a performance metric
num_splits = 3
c_values = np.logspace(-2, 0, 5)
# stratified k-fold
stratified_k_fold = StratifiedKFold(n_splits=num_splits)
scores = np.zeros(5)
nr_params = np.zeros(5)

In [ ]:
for train_data, valid_data in stratified_k_fold.split(x_train,
                                                      y_train):
    for i, c in enumerate(np.logspace(-2, 0, 5)):
        lr_classify = LogisticRegression(penalty='l2',
                                         class_weight='balanced',
                                         C = c, max_iter=1000)
        lr_classify.fit(x_train[train_data],
                        y_train[train_data])

        #validation_Set evaluation

        y_prediction = lr_classify.predict(x_train[valid_data])
        score_f1 = f1_score(y_train[valid_data],
                            y_prediction, average='weighted' )

        scores[i] += score_f1 / num_splits

        ### spot the selected parameters ##

        model_selected = SelectFromModel(lr_classify, prefit=True)
        nr_params[i] += np.sum(model_selected.get_support()) / num_splits

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(nr_params, scores)

for i, c in enumerate(c_values):
    plt.annotate(c, (nr_params[i], scores[i]))
plt.xlabel("Nr of parameters")
plt.ylabel("Avg F1 score")

In [ ]:
# XGBoost classifier
import xgboost
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


x_train, x_valid, y_train, y_valid = train_test_split(
    x_train,
    y_train,
    stratify=y_train,
    test_size=0.1)


xgb_clf = XGBClassifier(eval_metric="logloss")

xgb_clf.fit(x_train, y_train,
            eval_set=[(x_valid, y_valid)],
            verbose=True)

In [ ]:
y_pred = xgb_clf.predict(x_test)
predictions = [round(value) for value in y_pred]
print(classification_report(y_test,
                            predictions))

Parametertuning

Approach 1: Grid Search

In [ ]:
# Define helper function
def print_grid_search_metrics(gs):
    print ("Best score: " + str(gs.best_score_))
    print ("Best parameters set:")
    best_parameters = gs.best_params_
    for param_name in sorted(parameters.keys()):
        print(param_name + ':' + str(best_parameters[param_name]))

In [ ]:
# Choose the number of trees
parameters = {
    'n_estimators' : [80, 100, 120],
    'max_depth' : [4, 6, 8]
}
#Grid_xgb = GridSearchCV(xgb_clf, parameters, scoring='f1', cv=5)
Grid_xgb = GridSearchCV(xgb_clf, parameters, scoring='roc_auc', cv=5)
Grid_xgb.fit(x_train, y_train)

In [ ]:
# best number of parameters
print_grid_search_metrics(Grid_xgb)

In [ ]:
# best model
best_xgb_model_gs = Grid_xgb.best_estimator_

Approach 2: Bayesian Optimization

In [ ]:
pip install scikit-optimize

In [ ]:
from skopt import BayesSearchCV

In [ ]:
# Define the parameter search space
param_space = {
   'n_estimators' : (80, 100),
    'max_depth' : (4, 8)
}

In [ ]:
# Initialize Bayesian Optimization

#opt = BayesSearchCV(estimator=xgb_clf, search_spaces=param_space, scoring='f1', n_iter=32, cv=5, random_state=10)
opt = BayesSearchCV(estimator=xgb_clf, search_spaces=param_space, scoring='roc_auc', n_iter=32, cv=5, random_state=10)

# Fit the model
opt.fit(x_train, y_train)

In [ ]:
# best number of parameters
print("Best parameters found: ", opt.best_params_)
print("Best cross-validation score: ", opt.best_score_)

In [ ]:
# best model
best_xgb_model_by = opt.best_estimator_